In [ ]:
# Colab (celda):     !pip -q install google-genai
# VS Code (terminal): pip install google-genai python-dotenv ipykernel
import os, json
from google import genai
from google.genai import types
try:
    from google.colab import userdata          # Colab: llave en Secrets
    API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:                            # VS Code: llave en el archivo .env
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)
 
# Elegir un modelo "flash-lite" disponible en su cuenta
excluir = ("tts", "image", "live", "audio", "embedding", "preview", "exp")
modelos = [m.name.replace("models/", "") for m in client.models.list()
           if "generateContent" in (m.supported_actions or [])
           and "gemini" in m.name
           and not any(x in m.name for x in excluir)]
MODELO = (sorted(m for m in modelos if "flash-lite" in m)
          or sorted(m for m in modelos if "flash" in m) or modelos)[-1]
print("Modelo:", MODELO)


In [ ]:
PREGUNTA = "Un cliente deja US$25 de margen al mes y el churn mensual es 4% ¿Cuánto vale (CLV)?"
r = client.models.generate_content(model=MODELO, contents=PREGUNTA)
print(r.text)


In [ ]:
PREGUNTA = "Un cliente deja US$25 de margen al mes y el churn mensual es 4% y una tasa de descuento mensual del 0.8%. ¿Cuánto vale (CLV)?"
r = client.models.generate_content(model=MODELO, contents=PREGUNTA)
print(r.text)


In [ ]:
def calcular_clv(margen_mensual: float, churn_mensual: float) -> dict:
    """Calcula el CLV con la fórmula del curso:
    CLV = m × (1 − c) ÷ (c + d), con d = 0.008 (0.8% mensual).
    margen_mensual: margen por cliente al mes en US$.
    churn_mensual: proporción que se va cada mes (0.04 = 4%)."""
    m, c, d = margen_mensual, churn_mensual, 0.008
    return {"clv": round(m * (1 - c) / (c + d), 2)}
 
print(calcular_clv(25, 0.04))     # debe dar 500.0


## ¿Por qué el LLM da tres respuestas distintas?

**La respuesta correcta es US$500.** Sin herramienta, el LLM elige su propia fórmula, y la fórmula cambia según cómo le preguntemos.

**Datos:** m = margen mensual = US$25 · c = churn mensual (clientes que se van cada mes) = 4% = 0.04 · d = tasa de descuento mensual = 0.8% = 0.008

| Lo que le dimos | Fórmula que usó | Cuenta | Resultado |
|---|---|---|---|
| Solo m y c | m ÷ c | 25 ÷ 0.04 | **US$625** |
| m, c y también d en la pregunta | m ÷ (c + d) | 25 ÷ 0.048 | **US$520.83** |
| La herramienta `calcular_clv` | m × (1 − c) ÷ (c + d) | 24 ÷ 0.048 | **US$500** |

**¿Qué le faltó al LLM?**
- **(1 − c):** en el curso aprendimos que solo se cuenta al 96% de clientes que se queda desde el primer mes. El LLM se lo saltó las dos veces.
- **+ d:** un dólar en el futuro vale menos que hoy. En la primera respuesta también se lo saltó.

**Conclusión:** dar más datos en la pregunta **no garantiza** la fórmula correcta. La herramienta sí, porque la fórmula está escrita en código. **El LLM explica, el código calcula.**

In [ ]:
INSTRUCCIONES = """Eres un tutor de Marketing Analytics.
1. Para cualquier número, usa una herramienta. Nunca calcules de memoria.
2. Si falta un dato, pregunta; no lo supongas.
3. Explica en pasos cortos, muestra la fórmula y define cada sigla."""
CONFIG = types.GenerateContentConfig(system_instruction=INSTRUCCIONES,
    tools=[calcular_clv], temperature=0.2,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
r = client.models.generate_content(model=MODELO, contents=PREGUNTA,
                                   config=CONFIG)
for fc in r.function_calls or []:
    print("El LLM pide:", fc.name, dict(fc.args))


In [ ]:
HERRAMIENTAS = {"calcular_clv": calcular_clv}

def preguntar(pregunta, historial=None, max_pasos=5):
    historial = list(historial or []) + [types.Content(role="user", parts=[types.Part.from_text(text=pregunta)])]
    for paso in range(1, max_pasos + 1):
        r = client.models.generate_content(model=MODELO, contents=historial, config=CONFIG)
        historial.append(r.candidates[0].content)
        if not r.function_calls:                             # no pidió herramientas: respuesta final
            return r.text, historial
        partes = []
        for fc in r.function_calls:
            args = dict(fc.args or {})
            try:
                salida = HERRAMIENTAS[fc.name](**args)       # NUESTRO código ejecuta la herramienta
            except Exception as e:                           # Guardrail: si la pidió mal, no se cae
                salida = {"error": f"{type(e).__name__}: {e}. No inventes el dato: pídeselo al usuario."}
            print(f"Paso {paso}: pide {fc.name}({json.dumps(args)})  →  🔧 {salida}")
            partes.append(types.Part.from_function_response(name=fc.name, response={"resultado": salida}))
        historial.append(types.Content(role="user", parts=partes))
    return "Me detuve: llegué al máximo de pasos.", historial

respuesta, h = preguntar(PREGUNTA)
print("\n" + respuesta)


In [ ]:
r1, h = preguntar("¿Cuánto vale un cliente si el churn mensual es 4%?")
print(r1)


In [ ]:
r2, h = preguntar("El margen es US$40.", historial=h)
print(r2)

# Demo D · El agente de retención: del riesgo a la oferta
Le damos al agente una **meta**, no una pregunta. Él decide los pasos:
1. **Predice el riesgo** con el modelo campeón de churn de DataRobot.
2. **Lee el CRM** (Customer Relationship Management, la base de clientes).
3. **Redacta ofertas**, una distinta para cada cliente.
4. Una **persona aprueba** las ofertas caras.
5. Se **envían** (aquí simulado: se guardan en un archivo, no sale ningún correo real).
6. Se **revisa** quién respondió y lo aprendido vuelve al siguiente ciclo.

Requisito: haber corrido los pasos 1 a 5 (client, MODELO, types, json, os).

### Paso 8 · Conectar con DataRobot

In [ ]:
# Instalar una sola vez, en la terminal:  pip install datarobot datarobot-predict pandas
# En el archivo .env deben estar estas tres líneas:
#   DATAROBOT_API_TOKEN=su_llave_de_datarobot
#   DATAROBOT_ENDPOINT=https://app.datarobot.com/api/v2
#   DEPLOYMENT_ID=el_id_de_su_deployment_campeon
import os, json, time
import pandas as pd
from dotenv import load_dotenv

USAR_DATAROBOT = True      # Plan B: pongan False si no tienen deployment (el riesgo sale de una regla simple)

load_dotenv(override=True)    # vuelve a leer el .env, por si le agregaron líneas después del paso 1

if USAR_DATAROBOT:
    import datarobot as dr
    from datarobot_predict.deployment import predict

    dr.Client(token=os.getenv("DATAROBOT_API_TOKEN"), endpoint=os.getenv("DATAROBOT_ENDPOINT"))
    DEPLOYMENT = dr.Deployment.get(os.getenv("DEPLOYMENT_ID"))
    print("Conectado al modelo:", DEPLOYMENT.label)
# En Colab: cambien os.getenv("...") por userdata.get("...") y guarden las llaves en Secrets.

### Paso 9 · El CRM: los mismos datos con los que se entrenó el modelo

In [ ]:
# El modelo solo acepta las columnas con las que se entrenó, así que bajamos ese mismo archivo de DataRobot.
ARCHIVO_CRM = None    # si la descarga falla: bajen el archivo de DataRobot y pongan aquí su nombre, ej. "datos_churn.csv"

if USAR_DATAROBOT:
    proyecto = dr.Project.get(DEPLOYMENT.model["project_id"])
    TARGET = proyecto.target                 # la columna que el modelo predice, por ejemplo "churn"
    CLASE_CHURN = proyecto.positive_class    # el valor que significa "se fue", por ejemplo 1
    print("El modelo predice:", TARGET, "· clase positiva:", CLASE_CHURN)

    if ARCHIVO_CRM:
        datos = pd.read_csv(ARCHIVO_CRM)
    else:
        datos = proyecto.get_dataset().get_as_dataframe()

    # Separamos lo que de verdad pasó (1 = se fue, 0 = se quedó). El agente NO lo ve.
    se_fue = (datos[TARGET].astype(str) == str(CLASE_CHURN)).astype(int)
    datos = datos.drop(columns=[TARGET])

if "customer_id" not in datos.columns:            # si el archivo no trae un ID, le ponemos uno
    datos["customer_id"] = range(1, len(datos) + 1)

LO_QUE_PASO = dict(zip(datos["customer_id"], se_fue))   # cliente → 1 si se fue (para el paso 13)
CRM = datos.sample(min(200, len(datos)), random_state=7).reset_index(drop=True)
print("CRM listo:", len(CRM), "clientes")

# Predicción de prueba con 3 clientes, para ver qué devuelve el modelo
if USAR_DATAROBOT:
    prueba = predict(DEPLOYMENT, CRM.head(3)).dataframe
    print("Columnas que devuelve el modelo:", list(prueba.columns))
    for columna in prueba.columns:                # buscamos la columna de "probabilidad de irse"
        if columna.endswith("_PREDICTION") and f"_{CLASE_CHURN}_" in columna:
            COL_RIESGO = columna
    print("Usaremos la columna:", COL_RIESGO)

# Si la columna de arriba es la probabilidad de QUEDARSE (o los datos están al revés), pongan True:
INVERTIR = True

### Paso 10 · Calcular el riesgo de cada cliente (una sola vez)

In [ ]:
if USAR_DATAROBOT:
    predicciones = predict(DEPLOYMENT, CRM).dataframe       # le mandamos los 200 clientes al modelo
    CRM["riesgo"] = predicciones[COL_RIESGO].values          # probabilidad de irse, entre 0 y 1
    if INVERTIR:
        CRM["riesgo"] = 1 - CRM["riesgo"]


CRM["riesgo"] = CRM["riesgo"].round(3)
CRM.sort_values("riesgo", ascending=False).head()            # los 5 de más riesgo, para revisar

### Paso 10b · Las tres herramientas del agente

In [ ]:
OFERTAS = []               # aquí se van guardando las ofertas que propone el agente
LIMITE_DESCUENTO = 20      # Guardrail: descuento máximo que acepta el código (%)
APROBACION_DESDE = 10      # desde este descuento (%) decide una persona


def predecir_riesgo(n: int) -> dict:
    """Paso 1. Devuelve los n clientes con mayor riesgo de irse (riesgo de churn),
    según el modelo campeón de DataRobot. n: cuántos clientes (máximo 20)."""
    n = min(int(n), 20)
    top = CRM.sort_values("riesgo", ascending=False).head(n)
    clientes = []
    for _, fila in top.iterrows():
        clientes.append({"customer_id": int(fila["customer_id"]), "riesgo": float(fila["riesgo"])})
    return {"clientes": clientes}


def leer_crm(customer_ids: list[int]) -> dict:
    """Paso 2. Devuelve el historial de los clientes pedidos, con todas las columnas del CRM
    (por ejemplo segmento, ciudad, compras recientes, ticket promedio, antigüedad)."""
    historiales = []
    for cid in customer_ids:
        fila = CRM[CRM["customer_id"] == int(cid)]
        if len(fila) == 1:
            historiales.append(fila.fillna("").to_dict("records")[0])   # celdas vacías → ""
    if len(historiales) == 0:
        return {"error": "Ninguno de esos clientes está en el CRM."}
    return {"historiales": historiales}


def guardar_oferta(customer_id: int, descuento_pct: float, mensaje: str) -> dict:
    """Paso 3. Guarda la oferta de UN cliente.
    descuento_pct: descuento en % (por ejemplo 10).
    mensaje: el texto del correo, máximo 3 líneas, personalizado con su historial."""
    customer_id = int(customer_id)

    # Guardrail 1: el cliente tiene que existir
    if customer_id not in CRM["customer_id"].values:
        return {"error": f"El cliente {customer_id} no existe en el CRM. No inventes clientes."}

    # Guardrail 2: el descuento tiene que estar entre 1% y el límite
    if descuento_pct < 1 or descuento_pct > LIMITE_DESCUENTO:
        return {"error": f"El descuento debe estar entre 1% y {LIMITE_DESCUENTO}%."}

    # Guardrail 3: una sola oferta por cliente
    for oferta in OFERTAS:
        if oferta["customer_id"] == customer_id:
            return {"error": f"El cliente {customer_id} ya tiene oferta."}

    # ¿La aprueba una persona o pasa sola?
    if descuento_pct >= APROBACION_DESDE:
        estado = "pendiente_aprobacion"
    else:
        estado = "aprobada_automatica"

    OFERTAS.append({"customer_id": customer_id, "descuento_pct": descuento_pct,
                    "mensaje": mensaje, "estado": estado})
    return {"guardada": True, "estado": estado}


# El "menú" de herramientas: nombre que pide el LLM → función de Python que se ejecuta
HERR_RETENCION = {
    "predecir_riesgo": predecir_riesgo,
    "leer_crm": leer_crm,
    "guardar_oferta": guardar_oferta,
}

### Paso 11a · Dos funciones de ayuda: llamar a Gemini y el ciclo del agente

In [ ]:
from google.genai import errors


def llamar_gemini(contenido, config=None):
    """Llama a Gemini. Si nos pasamos del límite gratuito (error 429), espera 40 segundos y reintenta."""
    for intento in range(4):
        try:
            return client.models.generate_content(model=MODELO, contents=contenido, config=config)
        except errors.ClientError as e:
            if e.code == 429 and intento < 3:
                print("Límite de Gemini (15 llamadas por minuto). Espero 40 segundos…")
                time.sleep(40)
            else:
                raise


def ejecutar_agente(meta, config, herramientas, max_pasos=25):
    """El ciclo del agente: pensar → pedir herramienta → nuestro código la ejecuta → repetir."""
    historial = [types.Content(role="user", parts=[types.Part.from_text(text=meta)])]

    for paso in range(1, max_pasos + 1):
        # 1. El LLM piensa con todo lo que ha pasado hasta ahora
        r = llamar_gemini(historial, config)
        historial.append(r.candidates[0].content)

        # 2. Si no pidió herramientas, ya terminó: devolvemos su respuesta
        if not r.function_calls:
            return r.text

        # 3. Ejecutamos cada herramienta que pidió
        resultados = []
        for pedido in r.function_calls:
            datos = dict(pedido.args)
            try:
                salida = herramientas[pedido.name](**datos)
            except Exception as e:                 # Guardrail: si algo falla, no se cae; le avisamos
                salida = {"error": str(e)}
            print(f"Paso {paso}: {pedido.name}({str(datos)[:60]}) → {str(salida)[:70]}")
            resultados.append(types.Part.from_function_response(name=pedido.name,
                                                                response={"resultado": salida}))

        # 4. Le devolvemos los resultados y vuelve a pensar
        historial.append(types.Content(role="user", parts=resultados))

    return "Me detuve: llegué al máximo de pasos."

### Paso 11b · Instrucciones y configuración del agente de retención

In [ ]:
INSTRUCCIONES_RETENCION = """Eres un agente de retención de clientes de una cafetería.
Trabajas solo, sin preguntarle nada al usuario.
1. Usa predecir_riesgo para encontrar a los clientes con más riesgo de irse.
2. Usa leer_crm UNA vez con todos esos clientes para ver su historial.
3. Para CADA cliente usa guardar_oferta: un descuento y un mensaje de máximo 3 líneas
   que mencione algo de su historial. Da más descuento solo a quien más ha gastado.
   Pide TODOS los guardar_oferta en una sola respuesta (varias llamadas a la vez), no uno por uno.
4. Nunca inventes clientes ni datos. Al terminar, resume cuántas ofertas guardaste."""

CONFIG_RETENCION = types.GenerateContentConfig(
    system_instruction=INSTRUCCIONES_RETENCION,
    tools=[predecir_riesgo, leer_crm, guardar_oferta],
    temperature=0.2,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

### Paso 11c · El agente trabaja solo (pasos 1, 2 y 3 del flujo)

In [ ]:
OFERTAS.clear()          # empezamos con la bandeja vacía
resumen = ejecutar_agente("Meta: retener a los 10 clientes con mayor riesgo de irse.",
                          CONFIG_RETENCION, HERR_RETENCION)
print("\n" + resumen)
pd.DataFrame(OFERTAS)    # tabla con las ofertas que propuso

### Paso 12 · Una persona aprueba (paso 4) y se "envía" (paso 5)

In [ ]:
# Paso 4: revisamos una por una las ofertas que necesitan aprobación
for oferta in OFERTAS:
    if oferta["estado"] == "pendiente_aprobacion":
        print(f"\nCliente {oferta['customer_id']} · {oferta['descuento_pct']}% de descuento")
        print(oferta["mensaje"])
        respuesta = input("¿Aprobar? (s/n): ")
        if respuesta.strip().lower() == "s":
            oferta["estado"] = "aprobada"
        else:
            oferta["estado"] = "rechazada"

# Paso 5: "enviar" = guardar en un archivo. No sale ningún correo real.
enviadas = []
for oferta in OFERTAS:
    if oferta["estado"] in ("aprobada", "aprobada_automatica"):
        enviadas.append(oferta)

ENVIADAS = pd.DataFrame(enviadas)
ENVIADAS.to_csv("bandeja_de_salida.csv", index=False)
print(f"\n{len(ENVIADAS)} ofertas 'enviadas' → bandeja_de_salida.csv (no se envió ningún correo real)")

### Paso 13 · Revisar y ajustar (paso 6): lo aprendido vuelve al siguiente ciclo

In [ ]:
import numpy as np

if len(ENVIADAS) == 0:
    print("No se envió ninguna oferta: no hay nada que revisar.")
else:
    # ---- Pregunta 1: ¿acertó el modelo? (datos REALES del archivo, que el agente nunca vio)
    ENVIADAS["se_fue_real"] = ENVIADAS["customer_id"].map(LO_QUE_PASO)
    if USAR_DATAROBOT and INVERTIR:              # si los datos están al revés, lo real también
        ENVIADAS["se_fue_real"] = 1 - ENVIADAS["se_fue_real"]
    print(f"¿Acertó el modelo? De los clientes que marcó con riesgo, se fue el "
          f"{ENVIADAS['se_fue_real'].mean():.0%} (dato real).")

    # ---- Pregunta 2: ¿funcionó la oferta? (SIMULACIÓN, porque no enviamos correos de verdad)
    # simulacion: sin oferta se queda el 15% y cada punto de descuento suma 2 puntos.
    # Ejemplo: con 20% de descuento, la probabilidad de quedarse es 15% + 40% = 55%.
    azar = np.random.default_rng(42)

    def simular_se_queda(descuento_pct):
        probabilidad = 0.15 + 0.02 * descuento_pct
        if azar.random() < probabilidad:
            return 1
        return 0

    # Grupo con oferta: los que "recibieron" el correo
    con_oferta = []
    for descuento in ENVIADAS["descuento_pct"]:
        con_oferta.append(simular_se_queda(descuento))
    ENVIADAS["se_quedo_simulado"] = con_oferta

    # Grupo de control: los siguientes clientes de más riesgo, a los que NO les mandamos nada
    ordenados = CRM.sort_values("riesgo", ascending=False)
    control = ordenados[~ordenados["customer_id"].isin(ENVIADAS["customer_id"])].head(len(ENVIADAS))
    sin_oferta = []
    for _ in range(len(control)):
        sin_oferta.append(simular_se_queda(0))

    tasa_con = sum(con_oferta) / len(con_oferta)
    tasa_sin = sum(sin_oferta) / len(sin_oferta)
    print(f"¿Funcionó la oferta? (simulado) Con oferta se quedó el {tasa_con:.0%}; "
          f"sin oferta (grupo de control) el {tasa_sin:.0%}. Uplift = {tasa_con - tasa_sin:+.0%}")
    print(ENVIADAS[["customer_id", "descuento_pct", "se_quedo_simulado"]])

    # ---- El LLM lee los resultados y escribe 2 lecciones
    tabla = ENVIADAS[["customer_id", "descuento_pct", "se_quedo_simulado"]].to_string(index=False)
    pregunta = (f"Campaña de retención (resultados simulados).\n"
                f"Con oferta ({len(con_oferta)} clientes): se quedó el {tasa_con:.0%}.\n"
                f"Grupo de control sin oferta ({len(sin_oferta)} clientes): se quedó el {tasa_sin:.0%}.\n"
                f"Detalle de los que recibieron oferta (se_quedo_simulado: 1 = sí, 0 = no):\n{tabla}\n"
                "Escribe 2 lecciones cortas para el siguiente ciclo. No inventes datos que no estén aquí.")
    LECCIONES = llamar_gemini(pregunta).text
    print("\nLecciones para el siguiente ciclo:\n" + LECCIONES)